In [1]:
5+6

11

In [ ]:
from src.EDA1.Bivariate_Analysis.Statistics.num_num import num_num_eda
from src.config import BASE_DIR


num_num_eda(BASE_DIR / "src" / "Data" / "cleaned_ecommerce_dataset.csv")

2026-09-18 17:58:05,903 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Attempting to load dataset from: D:\Conversational_Data_Analytics\src\Data\cleaned_ecommerce_dataset.csv
2026-09-18 17:58:05,963 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Dataset successfully loaded with shape: (15000, 16)
2026-09-18 17:58:05,963 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Starting NUM-NUM exploratory data analysis...
2026-09-18 17:58:05,963 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Computing Pearson correlations for 7 numeric features...
2026-09-18 17:58:05,982 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Appended pearson correlation result for pair (Age, Quantity) | pearson_correlation: 0.0114, p_value: 1.6171e-01
2026-09-18 17:58:05,992 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_num: Appended pearson correlation result for pair (Age, Unit_Price) | pearson_correlation: -0.0165, p_value: 4.2743e-02
2026-09-18 17:58:06,002 [INFO] src.EDA1

In [5]:
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI
from src.config import BASE_DIR, config


# 1. Configuration for NVIDIA NIM using the OpenAI client
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=config.NVIDIA_API_KEY,
)
NVIDIA_MODEL = config.NVIDIA_MODEL

def summarize_existing_metrics_workbook(file_path: Path):
    """Reads worksheets containing pre-calculated metrics and sends them to NVIDIA NIM for summarization."""
    
    # Read all worksheets into a dictionary of DataFrames
    try:
        excel_data = pd.read_excel(file_path, sheet_name=None)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None

    workbook_text_context = ""

    # Loop through each worksheet containing the pre-calculated tables
    for sheet_name, df in excel_data.items():
        workbook_text_context += f"\n\n==============================\n"
        workbook_text_context += f"WORKSHEET / METRIC TABLE: {sheet_name}\n"
        workbook_text_context += f"==============================\n"
        # Convert the existing table data to markdown format
        workbook_text_context += df.to_markdown(index=False)

    # Construct the expert prompt for NVIDIA NIM
    prompt = f"""
    You are an expert quantitative data analyst. Below are the pre-calculated bivariate analysis 
    metrics (including Pearson, Spearman, Kendall correlations, and Mutual Information) extracted 
    directly from an Excel workbook's worksheets.

    Please provide a comprehensive executive summary that analyzes these metrics, highlighting:
    1. Strong linear relationships (Pearson).
    2. Monotonic or rank-based trends (Spearman & Kendall).
    3. Non-linear dependencies captured by Mutual Information.
    4. Key business insights, patterns, or anomalies observed across these sheets.

    Workbook Data:
    {workbook_text_context}
    """

    print("Sending pre-calculated metrics workbook data to NVIDIA NIM...")

    # Call the NVIDIA model using the OpenAI client
    try:
        response = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[
                {
                    "role": "system", 
                    "content": "You are a meticulous senior data-quality analyst and statistical metrics interpretation expert. You must use only supplied facts and must never fabricate statistics."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=2048,
        )
        
        return response.choices[0].message.content

    except Exception as e:
        print(f"NVIDIA API Error: {e}")
        return None

if __name__ == "__main__":
    # Replace with your actual Excel file path
    excel_file_path = Path(BASE_DIR / "src" / "EDA1" / "Bivariate_Analysis" / "Reports" / "num_num_eda.xlsx")
    
    if excel_file_path.exists():
        summary_result = summarize_existing_metrics_workbook(excel_file_path)
        if summary_result:
            print("\n=== NVIDIA NIM WORKBOOK METRICS SUMMARY ===\n")
            print(summary_result)
    else:
        print(f"File not found: {excel_file_path}")

Sending pre-calculated metrics workbook data to NVIDIA NIM...


2026-09-18 21:42:06,030 [INFO] httpx2: HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"



=== NVIDIA NIM WORKBOOK METRICS SUMMARY ===

**Executive Summary – Bivariate Relationships Across the Dataset**

---

### 1.  Strong Linear Relationships (Pearson)

| Pair                     | Pearson r | p‑value | Interpretation |
|--------------------------|----------|---------|----------------|
| **Unit_Price – Total_Amount** | **+0.835** | **0** (statistically significant) | A near‑perfect positive linear association: higher unit prices are tightly linked to higher total amounts. |
| **Quantity – Total_Amount**   | **+0.256** | 5.7 × 10⁻²²³ | A solid positive linear link; total amount rises as order quantity increases. |
| **Discount – Total_Amount**   | **‑0.041** | 4.4 × 10⁻⁷ | A modest but statistically significant negative linear trend; higher discount percentages correspond to slightly lower total amounts. |
| All other pairs | | | Correlations are ≤ |0.02| and p‑values > 0.1, indicating negligible linear association. |

**Key take‑away:** The only economically meaningful li

In [1]:
from src.Data.column_classification import classify_columns
from src.config import BASE_DIR

classify_columns(BASE_DIR / "src" / "Data" / "cleaned_ecommerce_dataset.csv")

{'univariate_candidates': {'numeric': ['Age',
   'Quantity',
   'Unit_Price',
   'Discount',
   'Rating',
   'Delivery_Days',
   'Total_Amount'],
  'categorical': ['Gender',
   'City',
   'Product_Category',
   'Payment_Method',
   'Returned'],
  'datetime': ['Order_Date']},
 'bivariate_candidates': {'numeric': ['Age',
   'Quantity',
   'Unit_Price',
   'Discount',
   'Rating',
   'Delivery_Days',
   'Total_Amount'],
  'categorical': ['Gender',
   'City',
   'Product_Category',
   'Payment_Method',
   'Returned'],
  'datetime': ['Order_Date']},
 'multivariate_candidates': {'numeric': ['Age',
   'Quantity',
   'Unit_Price',
   'Discount',
   'Rating',
   'Delivery_Days',
   'Total_Amount'],
  'categorical': ['Gender',
   'City',
   'Product_Category',
   'Payment_Method',
   'Returned'],
  'datetime': ['Order_Date']}}

In [2]:
from src.EDA1.Bivariate_Analysis.Statistics.num_cat import num_cat_eda
from src.config import BASE_DIR
num_cat_eda(file_path=BASE_DIR / "src" / "Data" / "cleaned_ecommerce_dataset.csv")

2026-09-18 21:34:03,021 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Attempting to load dataset from: D:\Conversational_Data_Analytics\src\Data\cleaned_ecommerce_dataset.csv
2026-09-18 21:34:03,069 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Dataset successfully loaded with shape: (15000, 16)
2026-09-18 21:34:03,075 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Starting NUM-CAT exploratory data analysis...
2026-09-18 21:34:03,079 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Computing group statistics for 7 numeric and 5 categorical features...
2026-09-18 21:34:03,092 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Appended group statistics for (Age, Gender), category: 'Female' | n: 7494, mean: 32.2745
2026-09-18 21:34:03,100 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_cat: Appended group statistics for (Age, Gender), category: 'Male' | n: 7067, mean: 32.3007
2026-09-18 21:34:03,104 [INFO] src.EDA1.Bivariate_Analysis.Statistics.num_ca

In [4]:
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI
from src.config import BASE_DIR, config

# Configuration for NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=config.NVIDIA_API_KEY,
)
NVIDIA_MODEL = config.NVIDIA_MODEL

def summarize_num_cat_workbook(file_path: Path | str) -> str:
    """Reads worksheets containing NUM-CAT EDA metrics and sends them to NVIDIA NIM for summarization."""
    
    file_path = Path(file_path)
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return None

    # Read all worksheets into a dictionary of DataFrames
    try:
        excel_data = pd.read_excel(file_path, sheet_name=None)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None

    workbook_text_context = ""

    # Loop through each worksheet containing the NUM-CAT metric tables
    for sheet_name, df in excel_data.items():
        workbook_text_context += f"\n\n==============================\n"
        workbook_text_context += f"METRIC SECTION: {sheet_name.upper()}\n"
        workbook_text_context += f"==============================\n"
        # Convert table data to markdown format to preserve layout for the LLM
        workbook_text_context += df.to_markdown(index=False)

    # Construct the expert prompt for NVIDIA NIM
    prompt = f"""
    You are a meticulous senior data-quality analyst and statistical interpretation expert. 
    Below are the pre-calculated Numerical vs. Categorical (NUM-CAT) exploratory data analysis metrics 
    extracted directly from an Excel workbook, covering group statistics, parametric/non-parametric tests 
    (Welch's t-test, Mann-Whitney U, Welch's ANOVA, Kruskal-Wallis), post-hoc analysis, and effect sizes.

    Please provide a comprehensive executive summary that analyzes these results, highlighting:
    1. Significant group differences between numerical features and categorical segments.
    2. Comparison of parametric vs. non-parametric findings (e.g., Welch's vs. Mann-Whitney / ANOVA vs. Kruskal-Wallis).
    3. Specific pairwise group insights from the post-hoc analysis.
    4. Practical significance based on the computed Effect Sizes.
    5. Key business implications or patterns observed.

    Workbook Data:
    {workbook_text_context}
    """

    print("Sending NUM-CAT workbook data to NVIDIA NIM for summarization...")

    # Call the NVIDIA model via OpenAI client
    try:
        response = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a meticulous senior data-quality analyst and statistical metrics "
                        "interpretation expert. You must use only supplied facts and must never "
                        "fabricate statistics."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=2048,
        )
        
        return response.choices[0].message.content

    except Exception as e:
        print(f"NVIDIA API Error: {e}")
        return None

if __name__ == "__main__":
    # Point this to the output workbook generated by your NUM-CAT pipeline
    output_workbook_path = Path(BASE_DIR / "src" / "EDA1" / "Bivariate_Analysis" / "Reports" / "num_cat_eda.xlsx")  
    
    summary_result = summarize_num_cat_workbook(output_workbook_path)
    if summary_result:
        print("\n=== NVIDIA NIM NUM-CAT EDA SUMMARY ===\n")
        print(summary_result)

Sending NUM-CAT workbook data to NVIDIA NIM for summarization...


2026-09-18 21:38:11,836 [INFO] httpx2: HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"



=== NVIDIA NIM NUM-CAT EDA SUMMARY ===

**Executive Summary – Data‑Quality & Business Insights (NUM‑CAT Exploration)**  

---

### 1. Significant Group Differences  

| Feature | Comparison | Statistical Test(s) | Significance | Interpretation |
|---------|------------|---------------------|--------------|----------------|
| **Age** | **City** | Welch’s ANOVA (p = 0.029) / Kruskal‑Wallis (p = 0.039) | **Significant** (p < 0.05) | Age varies modestly across cities (η² = 0.0014). The largest differences are observed between Ahmedabad/Bengaluru and the other cities, but the effect is small in practical terms. |
| **Age** | **Returned status** | Welch’s t‑test (p = 0.69) / Mann‑Whitney (p = 0.42) | Not significant | Age distribution is virtually identical for customers who returned an order vs. those who did not. |
| **Quantity** | **Gender** | Mann‑Whitney (p = 0.11) / Kruskal (p = 0.32) | Not significant | Order counts are similar for females, males and the “Other” group. |
| **Unit_Pri

In [1]:
from src.EDA1.Bivariate_Analysis.Statistics.cat_cat import cat_cat_eda
from src.config import BASE_DIR
cat_cat_eda(file_path=BASE_DIR / "src" / "Data" / "cleaned_ecommerce_dataset.csv")

2026-09-18 22:38:45,373 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Attempting to load dataset from: D:\Conversational_Data_Analytics\src\Data\cleaned_ecommerce_dataset.csv
2026-09-18 22:38:45,431 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Dataset successfully loaded with shape: (15000, 16)
2026-09-18 22:38:45,437 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Starting CAT-CAT exploratory data analysis...
2026-09-18 22:38:45,440 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Computing unified contingency summary for 5 categorical features...
2026-09-18 22:38:45,489 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Appended contingency summary results for pair (Gender, City)
2026-09-18 22:38:45,525 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Appended contingency summary results for pair (Gender, Product_Category)
2026-09-18 22:38:45,584 [INFO] src.EDA1.Bivariate_Analysis.Statistics.cat_cat: Appended contingency summary results for p

In [2]:
import os
import pandas as pd
from pathlib import Path
from openai import OpenAI
from src.config import BASE_DIR, config

# Configuration for NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=config.NVIDIA_API_KEY,
)
NVIDIA_MODEL = config.NVIDIA_MODEL

def summarize_cat_cat_workbook(file_path: Path | str) -> str:
    """Reads worksheets containing CAT-CAT EDA metrics and sends them to NVIDIA NIM for summarization."""
    
    file_path = Path(file_path)
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return None

    # Read all worksheets into a dictionary of DataFrames
    try:
        excel_data = pd.read_excel(file_path, sheet_name=None)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None

    workbook_text_context = ""

    # Loop through each worksheet containing the CAT-CAT metric tables
    for sheet_name, df in excel_data.items():
        workbook_text_context += f"\n\n==============================\n"
        workbook_text_context += f"METRIC SECTION: {sheet_name.upper()}\n"
        workbook_text_context += f"==============================\n"
        # Convert table data to markdown format to preserve layout for the LLM
        workbook_text_context += df.to_markdown(index=False)

    # Construct the expert prompt for NVIDIA NIM tailored for Categorical Analysis
    prompt = f"""
    You are a meticulous senior data-quality analyst and categorical association expert. 
    Below are the pre-calculated Categorical vs. Categorical (CAT-CAT) exploratory data analysis metrics 
    extracted directly from an Excel workbook, covering contingency table summaries, Chi-square tests of independence, 
    Cramer's V association effect sizes, and Fisher's exact tests (for sparse or low-frequency cells).

    Please provide a comprehensive executive summary that analyzes these results, highlighting:
    1. Statistically significant associations between categorical variables based on Chi-square and Fisher's exact p-values.
    2. Strength of association and practical significance interpreted through Cramer's V effect sizes.
    3. Standout frequency distributions, patterns, or cell-level imbalances identified in the contingency summaries.
    4. Data quality considerations (e.g., small sample sizes, sparse cells, or expected frequency violations requiring Fisher's exact test).
    5. Key business implications or behavioral patterns observed across the segments.

    Workbook Data:
    {workbook_text_context}
    """

    print("Sending CAT-CAT workbook data to NVIDIA NIM for summarization...")

    # Call the NVIDIA model via OpenAI client
    try:
        response = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a meticulous senior data-quality analyst and categorical association "
                        "interpretation expert. You must use only supplied facts and must never "
                        "fabricate statistics."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            max_tokens=2048,
        )
        
        return response.choices[0].message.content

    except Exception as e:
        print(f"NVIDIA API Error: {e}")
        return None

if __name__ == "__main__":
    # Point this to the output workbook generated by your CAT-CAT pipeline
    output_workbook_path = Path(BASE_DIR / "src" / "EDA1" / "Bivariate_Analysis" / "Reports" / "cat_cat_eda.xlsx")  
    
    summary_result = summarize_cat_cat_workbook(output_workbook_path)
    if summary_result:
        print("\n=== NVIDIA NIM CAT-CAT EDA SUMMARY ===\n")
        print(summary_result)

Sending CAT-CAT workbook data to NVIDIA NIM for summarization...


2026-09-18 22:46:45,894 [INFO] httpx2: HTTP Request: POST https://integrate.api.nvidia.com/v1/chat/completions "HTTP/1.1 200 OK"



=== NVIDIA NIM CAT-CAT EDA SUMMARY ===

**Executive Summary – Categorical Association Analysis**

**1. Statistically significant associations**  
- **City ↔ Payment_Method** (χ² = 74.88, df = 55, p = 0.0385; Cramer’s V = 0.0316) – the only pair that crosses the conventional 0.05 significance threshold.  
- **City ↔ Returned** (χ² = 20.90, df = 11, p = 0.0344; Cramer’s V = 0.0373) – also significant, indicating that the likelihood of a “Returned” outcome varies by city.  

All other chi‑square tests (Gender ↔ any variable, Product_Category ↔ any variable, Payment_Method ↔ Returned, etc.) have p‑values > 0.10, meaning no statistically detectable association.

**2. Strength of association (Cramer’s V)**  
- The highest association coefficients are observed for the two significant pairs:  
  - City ↔ Payment_Method Cramer’s V ≈ 0.032 (weak‑to‑moderate).  
  - City ↔ Returned    Cramer’s V ≈ 0.037 (weak‑to‑moderate).  
- All other Cramer’s V values are ≤ 0.025, reflecting very weak relatio